## Load the Model

In [1]:
import os
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import numpy as np
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
import wandb
import gradio as gr

root = Path('.')  # notebook is in TREES/
images_dir = root / "images"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

/opt/anaconda3/envs/SolutionsInPR/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [2]:
model_path = "models/Resnet18_V3.pth"

model = models.resnet18(weights=None)  # don't load pretrained weights
model.fc = nn.Linear(model.fc.in_features, 5)
state_dict = torch.load(model_path, map_location=torch.device(device))
model.load_state_dict(state_dict)
model.eval()

transform = Compose([
    Resize((224,224)), # Use 224x224
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406],         # Standard ImageNet mean
              std=[0.229, 0.224, 0.225])          # Standard ImageNet std dev
])
train_data = datasets.ImageFolder(images_dir, transform=transform)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

def predict(image):
    """
    Takes a PIL image, transforms it, and returns a dictionary of class confidences.
    """
    # The 'transform' object is the same one used for training
    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image_tensor)
        # Apply softmax to get probabilities
        probabilities = torch.nn.functional.softmax(outputs, dim=1)[0]

        # Create a dictionary of class: confidence
        confidences = {train_data.classes[i]: float(probabilities[i]) for i in range(len(train_data.classes))}

    return confidences

/var/folders/60/rs6s5kmn7rx3cwrh9tw6n05r0000gn/T/ipykernel_85155/1095370685.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_locat

In [3]:
examples = [
    ['lehmus/lehmus6.jpg'],
    ['pihlaja/pihlaja23.jpg'],
    ['koivu/koivu15.jpg']
]
examples = [[os.path.join(images_dir, path[0])] for path in examples]

iface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Upload Image"),
    outputs=gr.Label(num_top_classes=5, label="Predictions"),
    title="5-Class Image Classifier",
    description="Upload an image to see the model's prediction.",
    examples=examples
)

iface.launch(share=True, debug=True)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2025/10/20 11:30:03 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: i/o timeout


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> None


In [ ]:
import os
import random
from PIL import Image
import pandas as pd
import gradio as gr

# --- Omat polut ja esimerkit ---
images_dir = "images"  # muuta tarvittaessa
examples = [
    ['lehmus/lehmus6.jpg'],
    ['pihlaja/pihlaja23.jpg'],
    ['koivu/koivu15.jpg']
]
examples = [[os.path.join(images_dir, p[0])] for p in examples]

# --- Oletetaan että sinulla on tämä valmiina ---
# def predict(pil_image) -> dict[str, float]: ...
# Palauttaa esim: {"koivu": 0.81, "pihlaja": 0.12, "lehmus": 0.07}

# --- UI:n apufunktiot ---
def predict_ui(pil_image):
    """Kääre UI:lle: palauttaa sekä Label-dataa että taulukon top-5 riveistä."""
    if pil_image is None:
        return {}, pd.DataFrame(columns=["Luokka", "Todennäköisyys (%)"])
    raw = predict(pil_image)  # alkuperäinen malli
    # Järjestetään top-5 ja tehdään siisti taulukko
    items = sorted(raw.items(), key=lambda kv: kv[1], reverse=True)[:5]
    df = pd.DataFrame(
        [{"Luokka": k, "Todennäköisyys (%)": round(v * 100, 2)} for k, v in items]
    )
    # gr.Labelille voidaan antaa sama dict, jolloin se näyttää palkit
    return raw, df

def random_example():
    """Palauttaa satunnaisen esimerkkikuvan polun."""
    path = random.choice(examples)[0]
    return Image.open(path)

# --- Pieni, hillitty teema + CSS ---
theme = gr.themes.Soft(
    primary_hue="slate",
    neutral_hue="slate",
).set(
    body_background_fill="#0b1220",
    body_text_color="#e5e7eb",
    block_background_fill="#111827",
    block_shadow="0 10px 30px rgba(0,0,0,0.25)",
    input_background_fill="#0f172a"
)

custom_css = """
:root { --radius-xl: 16px; }
.gradio-container { max-width: 1100px !important; }
#app-title { font-size: 28px; font-weight: 700; letter-spacing: 0.3px; }
#app-sub { opacity: 0.85; }
.card { border-radius: 16px; border: 1px solid rgba(255,255,255,0.06); }
footer { opacity: 0.7; font-size: 12px; }
"""

# --- Rakennetaan näkymä ---
with gr.Blocks(theme=theme, css=custom_css, title="Puiden kuvantunnistus") as demo:
    gr.HTML("""
    <div style="display:flex; gap:16px; align-items:center; margin-top:8px;">
      <div>
        <div id="app-title">5-luokan kuvantunnistin</div>
        <div id="app-sub">Lataa kuva tai valitse esimerkki. Saat tulokseksi top-5-luokat ja todennäköisyydet.</div>
      </div>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=6, elem_classes=["card"]):
            image = gr.Image(
                type="pil",
                label="Lataa tai pudota kuva",
                height=360,
                show_label=True
            )
            with gr.Row():
                run_btn = gr.Button("Tunnista", variant="primary")
                rnd_btn = gr.Button("Satunnainen esimerkki")
                clear_btn = gr.Button("Tyhjennä")

        with gr.Column(scale=5, elem_classes=["card"]):
            label = gr.Label(
                num_top_classes=5,
                label="Ennusteet (palkit)"
            )
            table = gr.Dataframe(
                headers=["Luokka", "Todennäköisyys (%)"],
                label="Top-5 taulukkona",
                interactive=False,
                wrap=True
            )
            gr.Markdown(
                "Vinkki: saat parhaat tulokset terävillä, riittävän suurilla kuvilla. "
                "Jos käytät puhelinta, kuvaa kohde keskeltä ja hyvässä valossa."
            )

    gr.Examples(
        examples=examples,
        inputs=image,
        label="Esimerkkikuvat",
        examples_per_page=6
    )

    gr.HTML('<footer>© Gradio UI (Blocks). </footer>')

    # --- Toiminnot ---
    # Klikillä aja ennuste
    run_btn.click(predict_ui, inputs=image, outputs=[label, table])
    # Enter/submit kun kuva valmis
    gr.Examples(
        examples=examples,
        inputs=image,
        label="Esimerkkikuvat",
        examples_per_page=6,
        fn=predict_ui,                  # <-- lisää tämä
        outputs=[label, table]          # <-- ja tämä
    )
    # Tyhjennys
    clear_btn.click(lambda: (None, {}, pd.DataFrame(columns=["Luokka", "Todennäköisyys (%)"])),
                    outputs=[image, label, table])

# Käynnistys: localhostissa ei tarvitse share-linkkiä
demo.queue(default_concurrency_limit=2, max_size=None).launch(
    server_name="127.0.0.1",
    server_port=7860,
    share=False,
    debug=True,
    favicon_path=None
)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
